# Pyannote Diarization testing - Speaker Diarization with Pyannote on VAST
### <https://vast.ai/article/speaker-diarization-with-pyannote-on-vast>
### <https://github.com/pyannote/pyannote-audio>

In [1]:
# user configurable settings
audio_file = "dummy_data/sample_audio.mp3"


In [2]:
# setup stuff

from pydantic_settings import BaseSettings, SettingsConfigDict

class NotebookSettings(BaseSettings):
    hf_token: str
    model_config = SettingsConfigDict(env_file='.env', env_file_encoding='utf-8')

settings = NotebookSettings()

#print(f"HF_TOKEN env variable: {settings.hf_token}")


## Introduction

This notebook demonstrates how to implement Speaker Diarization using the Pyannote Audio library on VAST.ai's cloud computing platform. Speaker Diarization is the process of partitioning an audio stream into segments according to the speaker identity, answering the question "who spoke when?"

### Why Speaker Diarization Matters

Speaker Diarization provides several key benefits for audio processing pipelines:

1. **Speaker Identification**: It identifies different speakers in a conversation, meeting, or any multi-speaker audio recording.

2. **Improved Transcription**: When combined with speech-to-text systems, diarization allows for speaker-attributed transcripts, making it clear who said what.

3. **Processing Efficiency**: By segmenting audio by speaker and removing non-speech portions, diarization can significantly reduce the computational load for downstream tasks like speech recognition, allowing these systems to process only relevant speech segments rather than the entire audio file.

4. **Audio Indexing**: Makes audio content searchable by speaker, allowing users to find all segments where a specific person speaks.


### What This Notebook Does

In this notebook, we will:
- Set up the Pyannote Audio Speaker Diarization pipeline
- Process audio files to detect different speakers and their speaking turns
- Calculate speaking time for each identified speaker
- Identify regions with overlapping speech
- Extract and save speaker-specific segments from the input audio
- Play and verify the diarization results

The output will be a collection of audio files separated by speaker, making them ready for further processing in speech-to-text pipelines or speaker-specific analysis.


## Choosing an Instance

For running the Pyannote Speaker Diarization model on VAST.ai, you'll need a relatively modest GPU setup. The pyannote/speaker-diarization-3.1 model runs in pure PyTorch and is designed to be efficient. Here are the recommended specifications:

- GPU: A low-end GPU like an RTX 3060 or 4060 would be sufficient.
- VRAM: 6-8GB of VRAM should be adequate as the Pyannote diarization pipeline is relatively efficient.
- RAM: 8-16GB system RAM is recommended for processing audio files.
- Storage: At least 10GB for the model, dependencies, and your audio files.
- CUDA: Make sure the instance has CUDA installed (version 11.0+ recommended).
- Python: Python 3.8+ with PyTorch installed.


## Install Dependencies

In [ ]:
%%sh
uv add pyannote.audio
uv add pydub
#uv add librosa # this is the package that barfs out the below error
uv add datasets

# TODO fix this:
# $ uv add librosa
# Resolved 341 packages in 1.08s
#   x Failed to build `numba==0.53.1`                                               |-> The build backend returned an error                                         `-> Call to `setuptools.build_meta:__legacy__.build_wheel` failed (exit
#       code: 1)

#       [stderr]
#       D:\Users\andrew.sparkes\AppData\Local\uv\cache\sdists-v9\pypi\numba\0.53.1\-XCj9CkqdlASAdpG03mAN\src\versioneer.py:415:
#       SyntaxWarning: "\s" is an invalid escape sequence. Such sequences will
#       not work in the future. Did you mean "\\s"? A raw string is also an
#       option.
#         mo = re.search(r'=\s*"(.*)"', line)
#       Traceback (most recent call last):
#         File "<string>", line 14, in <module>
#           requires = get_requires_for_build({})
#         File
#       "D:\Users\andrew.sparkes\AppData\Local\uv\cache\builds-v0\.tmp0vRXFF\Lib\site-packages\setuptools\build_meta.py",
#       line 333, in get_requires_for_build_wheel
#           return self._get_build_requires(config_settings, requirements=[])
#                  ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#         File
#       "D:\Users\andrew.sparkes\AppData\Local\uv\cache\builds-v0\.tmp0vRXFF\Lib\site-packages\setuptools\build_meta.py",
#       line 301, in _get_build_requires
#           self.run_setup()
#           ~~~~~~~~~~~~~~^^
#         File
#       "D:\Users\andrew.sparkes\AppData\Local\uv\cache\builds-v0\.tmp0vRXFF\Lib\site-packages\setuptools\build_meta.py",
#       line 520, in run_setup
#           super().run_setup(setup_script=setup_script)
#           ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^
#         File
#       "D:\Users\andrew.sparkes\AppData\Local\uv\cache\builds-v0\.tmp0vRXFF\Lib\site-packages\setuptools\build_meta.py",
#       line 317, in run_setup
#           exec(code, locals())
#           ~~~~^^^^^^^^^^^^^^^^
#         File "<string>", line 50, in <module>
#         File "<string>", line 47, in _guard_py_ver
#       RuntimeError: Cannot install on Python version 3.14.2; only versions
#       >=3.6,<3.10 are supported.

#       hint: This usually indicates a problem with the package or the build
#       environment.
#   help: If you want to add the package regardless of the failed resolution,
#         provide the `--frozen` flag to skip locking and syncing.



Couldn't find program: 'sh'


In [ ]:
%%sh
apt-get update && apt-get install -y ffmpeg
# TODO use something like chocolatey or other pkg manager for win64?


bash: line 1: apt-get: command not found


CalledProcessError: Command 'b'apt-get update && apt-get install -y ffmpeg\n'' returned non-zero exit status 127.

## Set up your Huggingface Token

Here we set our huggingface token as `HF_TOKEN`. We need this to access the model.

Ensure that you have accepted the terms for https://huggingface.co/pyannote/speaker-diarization-3.1 and https://huggingface.co/pyannote/segmentation-3.0. This model is free to use, but you must accept their terms.

In [5]:
# Make sure you've accepted the user conditions at:
# https://huggingface.co/pyannote/speaker-diarization-3.1
# https://huggingface.co/pyannote/segmentation-3.0

HF_TOKEN = settings.hf_token


## Download Test Data

We will use a sample file from the AMI Meeting Corpus dataset https://huggingface.co/datasets/diarizers-community/ami, which is a collection of 100 hours of meeting recordings.

This code efficiently pulls a few sample files from the dataset. If you want to download the entire dataset there are better methods - see the Huggingface API.

In [ ]:
from datasets import load_dataset
import os
import soundfile as sf

# Create a directory to save the files
os.makedirs("ami_samples", exist_ok = True)

# Load the dataset with the correct split
dataset = load_dataset("diarizers-community/ami", "ihm", split = "train", streaming = True)


# load any number of samples
n_samples = 1
samples = list(dataset.take(n_samples))

for i, sample in enumerate(samples):

    audio = sample["audio"]
    audio_array = audio["array"]
    sampling_rate = audio["sampling_rate"]
    
    # Calculate duration in seconds
    duration = len(audio_array) / sampling_rate
    
    # Use soundfile to save the audio
    output_path = f"ami_samples/sample_{i}.wav"
    sf.write(output_path, audio_array, sampling_rate)
    
    print(f"Saved {output_path} - Speaker: {sample['speakers']} - Duration: {duration:.2f} seconds")

# so far, it spits out the following:
# z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\multiprocess\connection.py:335: SyntaxWarning: 'return' in a 'finally' block
#   return f
# z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\multiprocess\connection.py:337: SyntaxWarning: 'return' in a 'finally' block
#   return self._get_more_data(ov, maxsize)

# outputs html widgets for the following:
# README.md:  3.90k/? [00:00<00:00, 696kB/s]

# Resolving data files: 100% 19/19 [00:00<00:00, 3306.85it/s]

# Resolving data files: 100% 19/19 [00:00<00:00, 4778.54it/s]

# then, it finally chokes and dies with:
# ---------------------------------------------------------------------------
# RuntimeError                              Traceback (most recent call last)
# Cell In[6], line 14
#      12 # load any number of samples
#      13 n_samples = 1
# ---> 14 samples = list(dataset.take(n_samples))
#      16 for i, sample in enumerate(samples):
#      18     audio = sample["audio"]

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\datasets\iterable_dataset.py:2543, in IterableDataset.__iter__(self)
#    2540         yield formatter.format_row(pa_table)
#    2541     return
# -> 2543 for key, example in ex_iterable:
#    2544     # no need to format thanks to FormattedExamplesIterable
#    2545     yield example

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\datasets\iterable_dataset.py:2061, in FormattedExamplesIterable.__iter__(self)
#    2058 if self.ex_iterable.iter_arrow:
#    2059     # feature casting (inc column addition) handled within self._iter_arrow()
#    2060     for key, pa_table in self._iter_arrow():
# -> 2061         batch = formatter.format_batch(pa_table)
#    2062         for example in _batch_to_examples(batch):
#    2063             yield key, example

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\datasets\formatting\formatting.py:472, in PythonFormatter.format_batch(self, pa_table)
#     470     return LazyBatch(pa_table, self)
#     471 batch = self.python_arrow_extractor().extract_batch(pa_table)
# --> 472 batch = self.python_features_decoder.decode_batch(batch)
#     473 return batch

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\datasets\formatting\formatting.py:234, in PythonFeaturesDecoder.decode_batch(self, batch)
#     233 def decode_batch(self, batch: dict) -> dict:
# --> 234     return self.features.decode_batch(batch, token_per_repo_id=self.token_per_repo_id) if self.features else batch

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\datasets\features\features.py:2161, in Features.decode_batch(self, batch, token_per_repo_id)
#    2157 decoded_batch = {}
#    2158 for column_name, column in batch.items():
#    2159     decoded_batch[column_name] = (
#    2160         [
# -> 2161             decode_nested_example(self[column_name], value, token_per_repo_id=token_per_repo_id)
#    2162             if value is not None
#    2163             else None
#    2164             for value in column
#    2165         ]
#    2166         if self._column_requires_decoding[column_name]
#    2167         else column
#    2168     )
#    2169 return decoded_batch

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\datasets\features\features.py:1419, in decode_nested_example(schema, obj, token_per_repo_id)
#    1416 # Object with special decoding:
#    1417 elif hasattr(schema, "decode_example") and getattr(schema, "decode", True):
#    1418     # we pass the token to read and decode files from private repositories in streaming mode
# -> 1419     return schema.decode_example(obj, token_per_repo_id=token_per_repo_id) if obj is not None else None
#    1420 return obj

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\datasets\features\audio.py:184, in Audio.decode_example(self, value, token_per_repo_id)
#     167 """Decode example audio file into audio data.
#     168 
#     169 Args:
#    (...)    181     `torchcodec.decoders.AudioDecoder`
#     182 """
#     183 if config.TORCHCODEC_AVAILABLE:
# --> 184     from ._torchcodec import AudioDecoder
#     185 else:
#     186     raise ImportError("To support decoding audio data, please install 'torchcodec'.")

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\datasets\features\_torchcodec.py:2
#       1 import numpy as np
# ----> 2 from torchcodec.decoders import AudioDecoder as _AudioDecoder
#       5 class AudioDecoder(_AudioDecoder):
#       6     def __getitem__(self, key: str):

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\__init__.py:12
#       9 # Note: usort wants to put Frame and FrameBatch after decoders and samplers,
#      10 # but that results in circular import.
#      11 from ._frame import AudioSamples, Frame, FrameBatch  # usort:skip # noqa
# ---> 12 from . import decoders, encoders, samplers, transforms  # noqa
#      14 try:
#      15     # Note that version.py is generated during install.
#      16     from .version import __version__  # noqa: F401

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\decoders\__init__.py:7
#       1 # Copyright (c) Meta Platforms, Inc. and affiliates.
#       2 # All rights reserved.
#       3 #
#       4 # This source code is licensed under the BSD-style license found in the
#       5 # LICENSE file in the root directory of this source tree.
# ----> 7 from .._core import AudioStreamMetadata, VideoStreamMetadata
#       8 from ._audio_decoder import AudioDecoder  # noqa
#       9 from ._decoder_utils import set_cuda_backend  # noqa

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\__init__.py:8
#       1 # Copyright (c) Meta Platforms, Inc. and affiliates.
#       2 # All rights reserved.
#       3 #
#       4 # This source code is licensed under the BSD-style license found in the
#       5 # LICENSE file in the root directory of this source tree.
# ----> 8 from ._metadata import (
#       9     AudioStreamMetadata,
#      10     ContainerMetadata,
#      11     get_container_metadata,
#      12     get_container_metadata_from_header,
#      13     VideoStreamMetadata,
#      14 )
#      15 from .ops import (
#      16     _add_video_stream,
#      17     _get_backend_details,
#    (...)     45     seek_to_pts,
#      46 )

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\_metadata.py:16
#      12 from fractions import Fraction
#      14 import torch
# ---> 16 from torchcodec._core.ops import (
#      17     _get_container_json_metadata,
#      18     _get_stream_json_metadata,
#      19     create_from_file,
#      20 )
#      23 SPACES = "  "
#      26 @dataclass
#      27 class StreamMetadata:

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py:109
#     105             return os.add_dll_directory(str(ffmpeg_dir))  # that's the actual CM
#     108 with expose_ffmpeg_dlls():
# --> 109     ffmpeg_major_version, core_library_path = load_torchcodec_shared_libraries()
#     112 # Note: We use disallow_in_graph because PyTorch does constant propagation of
#     113 # factory functions.
#     114 create_from_file = torch._dynamo.disallow_in_graph(
#     115     torch.ops.torchcodec_ns.create_from_file.default
#     116 )

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py:76, in load_torchcodec_shared_libraries()
#      69         exceptions.append((ffmpeg_major_version, exc_traceback))
#      71 traceback_info = (
#      72     "\n[start of libtorchcodec loading traceback]\n"
#      73     + "\n".join(f"FFmpeg version {v}:\n{tb}" for v, tb in exceptions)
#      74     + "[end of libtorchcodec loading traceback]."
#      75 )
# ---> 76 raise RuntimeError(
#      77     f"""Could not load libtorchcodec. Likely causes:
#      78       1. FFmpeg is not properly installed in your environment. We support
#      79          versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
#      80          for each of those versions. Errors for versions not installed on
#      81          your system are expected; only the error for your installed FFmpeg
#      82          version is relevant. On Windows, ensure you've installed the
#      83          "full-shared" version which ships DLLs.
#      84       2. The PyTorch version ({torch.__version__}) is not compatible with
#      85          this version of TorchCodec. Refer to the version compatibility
#      86          table:
#      87          https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
#      88       3. Another runtime dependency; see exceptions below.
#      89 
#      90     The following exceptions were raised as we tried to load libtorchcodec:
#      91     """
#      92     f"{traceback_info}"
#      93 )

# RuntimeError: Could not load libtorchcodec. Likely causes:
#           1. FFmpeg is not properly installed in your environment. We support
#              versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
#              for each of those versions. Errors for versions not installed on
#              your system are expected; only the error for your installed FFmpeg
#              version is relevant. On Windows, ensure you've installed the
#              "full-shared" version which ships DLLs.
#           2. The PyTorch version (2.10.0+cpu) is not compatible with
#              this version of TorchCodec. Refer to the version compatibility
#              table:
#              https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
#           3. Another runtime dependency; see exceptions below.

#         The following exceptions were raised as we tried to load libtorchcodec:
        
# [start of libtorchcodec loading traceback]
# FFmpeg version 8:
# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1442, in load_library
#     ctypes.CDLL(path)
#     ~~~~~~~~~~~^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 433, in __init__
#     self._handle = self._load_library(name, mode, handle, winmode)
#                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 451, in _load_library
#     return _LoadLibrary(self._name, winmode)
# FileNotFoundError: Could not find module '\\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core8.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# The above exception was the direct cause of the following exception:

# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py", line 57, in load_torchcodec_shared_libraries
#     torch.ops.load_library(core_library_path)
#     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1444, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: \\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core8.dll

# FFmpeg version 7:
# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1442, in load_library
#     ctypes.CDLL(path)
#     ~~~~~~~~~~~^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 433, in __init__
#     self._handle = self._load_library(name, mode, handle, winmode)
#                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 451, in _load_library
#     return _LoadLibrary(self._name, winmode)
# FileNotFoundError: Could not find module '\\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core7.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# The above exception was the direct cause of the following exception:

# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py", line 57, in load_torchcodec_shared_libraries
#     torch.ops.load_library(core_library_path)
#     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1444, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: \\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core7.dll

# FFmpeg version 6:
# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1442, in load_library
#     ctypes.CDLL(path)
#     ~~~~~~~~~~~^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 433, in __init__
#     self._handle = self._load_library(name, mode, handle, winmode)
#                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 451, in _load_library
#     return _LoadLibrary(self._name, winmode)
# FileNotFoundError: Could not find module '\\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core6.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# The above exception was the direct cause of the following exception:

# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py", line 57, in load_torchcodec_shared_libraries
#     torch.ops.load_library(core_library_path)
#     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1444, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: \\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core6.dll

# FFmpeg version 5:
# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1442, in load_library
#     ctypes.CDLL(path)
#     ~~~~~~~~~~~^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 433, in __init__
#     self._handle = self._load_library(name, mode, handle, winmode)
#                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 451, in _load_library
#     return _LoadLibrary(self._name, winmode)
# FileNotFoundError: Could not find module '\\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core5.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# The above exception was the direct cause of the following exception:

# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py", line 57, in load_torchcodec_shared_libraries
#     torch.ops.load_library(core_library_path)
#     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1444, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: \\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core5.dll

# FFmpeg version 4:
# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1442, in load_library
#     ctypes.CDLL(path)
#     ~~~~~~~~~~~^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 433, in __init__
#     self._handle = self._load_library(name, mode, handle, winmode)
#                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 451, in _load_library
#     return _LoadLibrary(self._name, winmode)
# FileNotFoundError: Could not find module '\\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core4.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# The above exception was the direct cause of the following exception:

# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py", line 57, in load_torchcodec_shared_libraries
#     torch.ops.load_library(core_library_path)
#     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1444, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: \\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core4.dll
# [end of libtorchcodec loading traceback].


z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\multiprocess\connection.py:335: SyntaxWarning: 'return' in a 'finally' block
  return f
z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\multiprocess\connection.py:337: SyntaxWarning: 'return' in a 'finally' block
  return self._get_more_data(ov, maxsize)


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/19 [00:00<?, ?it/s]

RuntimeError: Could not load libtorchcodec. Likely causes:
          1. FFmpeg is not properly installed in your environment. We support
             versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
             for each of those versions. Errors for versions not installed on
             your system are expected; only the error for your installed FFmpeg
             version is relevant. On Windows, ensure you've installed the
             "full-shared" version which ships DLLs.
          2. The PyTorch version (2.10.0+cpu) is not compatible with
             this version of TorchCodec. Refer to the version compatibility
             table:
             https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
          3. Another runtime dependency; see exceptions below.

        The following exceptions were raised as we tried to load libtorchcodec:
        
[start of libtorchcodec loading traceback]
FFmpeg version 8:
Traceback (most recent call 

## Speaker Diarization

First we set up the Speaker Diarization pipeline.

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained( # this call causes the giant error below
        "pyannote/speaker-diarization-3.1",
        token =HF_TOKEN,
)

# Move pipeline to appropriate device
pipeline = pipeline.to(device)

# giant error from pipeline call on line 7:
# z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\pyannote\audio\core\io.py:47: UserWarning: 
# torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
# * use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
# * fix torchcodec installation. Error message was:

# Could not load libtorchcodec. Likely causes:
#           1. FFmpeg is not properly installed in your environment. We support
#              versions 4, 5, 6, 7, and 8, and we attempt to load libtorchcodec
#              for each of those versions. Errors for versions not installed on
#              your system are expected; only the error for your installed FFmpeg
#              version is relevant. On Windows, ensure you've installed the
#              "full-shared" version which ships DLLs.
#           2. The PyTorch version (2.10.0+cpu) is not compatible with
#              this version of TorchCodec. Refer to the version compatibility
#              table:
#              https://github.com/pytorch/torchcodec?tab=readme-ov-file#installing-torchcodec.
#           3. Another runtime dependency; see exceptions below.

#         The following exceptions were raised as we tried to load libtorchcodec:
        
# [start of libtorchcodec loading traceback]
# FFmpeg version 8:
# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1442, in load_library
#     ctypes.CDLL(path)
#     ~~~~~~~~~~~^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 433, in __init__
#     self._handle = self._load_library(name, mode, handle, winmode)
#                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 451, in _load_library
#     return _LoadLibrary(self._name, winmode)
# FileNotFoundError: Could not find module '\\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core8.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# The above exception was the direct cause of the following exception:

# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py", line 57, in load_torchcodec_shared_libraries
#     torch.ops.load_library(core_library_path)
#     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1444, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: \\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core8.dll

# FFmpeg version 7:
# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1442, in load_library
#     ctypes.CDLL(path)
#     ~~~~~~~~~~~^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 433, in __init__
#     self._handle = self._load_library(name, mode, handle, winmode)
#                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 451, in _load_library
#     return _LoadLibrary(self._name, winmode)
# FileNotFoundError: Could not find module '\\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core7.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# The above exception was the direct cause of the following exception:

# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py", line 57, in load_torchcodec_shared_libraries
#     torch.ops.load_library(core_library_path)
#     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1444, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: \\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core7.dll

# FFmpeg version 6:
# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1442, in load_library
#     ctypes.CDLL(path)
#     ~~~~~~~~~~~^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 433, in __init__
#     self._handle = self._load_library(name, mode, handle, winmode)
#                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 451, in _load_library
#     return _LoadLibrary(self._name, winmode)
# FileNotFoundError: Could not find module '\\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core6.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# The above exception was the direct cause of the following exception:

# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py", line 57, in load_torchcodec_shared_libraries
#     torch.ops.load_library(core_library_path)
#     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1444, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: \\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core6.dll

# FFmpeg version 5:
# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1442, in load_library
#     ctypes.CDLL(path)
#     ~~~~~~~~~~~^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 433, in __init__
#     self._handle = self._load_library(name, mode, handle, winmode)
#                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 451, in _load_library
#     return _LoadLibrary(self._name, winmode)
# FileNotFoundError: Could not find module '\\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core5.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# The above exception was the direct cause of the following exception:

# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py", line 57, in load_torchcodec_shared_libraries
#     torch.ops.load_library(core_library_path)
#     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1444, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: \\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core5.dll

# FFmpeg version 4:
# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1442, in load_library
#     ctypes.CDLL(path)
#     ~~~~~~~~~~~^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 433, in __init__
#     self._handle = self._load_library(name, mode, handle, winmode)
#                    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
#   File "D:\Users\andrew.sparkes\AppData\Roaming\uv\python\cpython-3.14.2-windows-x86_64-none\Lib\ctypes\__init__.py", line 451, in _load_library
#     return _LoadLibrary(self._name, winmode)
# FileNotFoundError: Could not find module '\\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core4.dll' (or one of its dependencies). Try using the full path with constructor syntax.

# The above exception was the direct cause of the following exception:

# Traceback (most recent call last):
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\_core\ops.py", line 57, in load_torchcodec_shared_libraries
#     torch.ops.load_library(core_library_path)
#     ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
#   File "z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\_ops.py", line 1444, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: \\zdrive.labs.cset.oit.edu\zdrive\andrew.sparkes\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torchcodec\libtorchcodec_core4.dll
# [end of libtorchcodec loading traceback].
#   warnings.warn(

# it wasn't satisfied with just that giant dump of errors, so it appended one more, thankfully an easy fix...
# ---------------------------------------------------------------------------
# TypeError                                 Traceback (most recent call last)
# Cell In[7], line 7
#       3 device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#       5 from pyannote.audio import Pipeline
# ----> 7 pipeline = Pipeline.from_pretrained(
#       8         "pyannote/speaker-diarization-3.1",
#       9         use_auth_token = HF_TOKEN
#      10 )
#      12 # Move pipeline to appropriate device
#      13 pipeline = pipeline.to(device)

# TypeError: Pipeline.from_pretrained() got an unexpected keyword argument 'use_auth_token'

# k...

# changed named parameter from use_auth_token to token
# and it worked! Kinda!
# New output:
# config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]
# pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]
# ---------------------------------------------------------------------------
# UnpicklingError                           Traceback (most recent call last)
# Cell In[9], line 7
#       3 device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#       5 from pyannote.audio import Pipeline
# ----> 7 pipeline = Pipeline.from_pretrained( # this call causes the giant error below
#       8         "pyannote/speaker-diarization-3.1",
#       9         token = HF_TOKEN,
#      10 )
#      12 # Move pipeline to appropriate device
#      13 pipeline = pipeline.to(device)

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\pyannote\audio\core\pipeline.py:244, in Pipeline.from_pretrained(cls, checkpoint, revision, hparams_file, token, cache_dir)
#     242 params.setdefault("token", token)
#     243 params.setdefault("cache_dir", cache_dir)
# --> 244 pipeline = Klass(**params)
#     246 # save pipeline origin (HF, local, etc) and class name as attributes for telemetry purposes
#     247 pipeline._otel_origin = otel_origin

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\pyannote\audio\pipelines\speaker_diarization.py:222, in SpeakerDiarization.__init__(self, legacy, segmentation, segmentation_step, embedding, embedding_exclude_overlap, plda, clustering, embedding_batch_size, segmentation_batch_size, der_variant, token, cache_dir)
#     219 self.legacy = legacy
#     221 self.segmentation_model = segmentation
# --> 222 model: Model = get_model(segmentation, token=token, cache_dir=cache_dir)
#     224 self.segmentation_step = segmentation_step
#     226 self.embedding = embedding

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\pyannote\audio\pipelines\utils\getter.py:115, in get_model(model, token, cache_dir)
#     112     pass
#     114 elif isinstance(model, str):
# --> 115     _model = Model.from_pretrained(
#     116         model,
#     117         token=token,
#     118         cache_dir=cache_dir,
#     119         strict=False,
#     120     )
#     121     if _model:
#     122         model = _model

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\pyannote\audio\core\model.py:602, in Model.from_pretrained(cls, checkpoint, map_location, strict, subfolder, revision, token, cache_dir, **kwargs)
#     599     map_location = default_map_location
#     601 # load checkpoint using lightning
# --> 602 loaded_checkpoint = pl_load(path_to_model_checkpoint, map_location=map_location)
#     604 # check that the checkpoint is compatible with the current version
#     605 versions = loaded_checkpoint["pyannote.audio"]["versions"]

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\lightning\fabric\utilities\cloud_io.py:73, in _load(path_or_url, map_location, weights_only)
#      71 fs = get_filesystem(path_or_url)
#      72 with fs.open(path_or_url, "rb") as f:
# ---> 73     return torch.load(
#      74         f,
#      75         map_location=map_location,  # type: ignore[arg-type]
#      76         weights_only=weights_only,
#      77     )

# File z:\code\STAT405_AudioNotetaker\.venv\Lib\site-packages\torch\serialization.py:1548, in load(f, map_location, pickle_module, weights_only, mmap, **pickle_load_args)
#    1540                 return _load(
#    1541                     opened_zipfile,
#    1542                     map_location,
#    (...)   1545                     **pickle_load_args,
#    1546                 )
#    1547             except pickle.UnpicklingError as e:
# -> 1548                 raise pickle.UnpicklingError(_get_wo_message(str(e))) from None
#    1549         return _load(
#    1550             opened_zipfile,
#    1551             map_location,
#    (...)   1554             **pickle_load_args,
#    1555         )
#    1556 if mmap:

# UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, do those steps only if you trust the source of the checkpoint. 
# 	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
# 	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
# 	WeightsUnpickler error: Unsupported global: GLOBAL torch.torch_version.TorchVersion was not an allowed global by default. Please use `torch.serialization.add_safe_globals([torch.torch_version.TorchVersion])` or the `torch.serialization.safe_globals([torch.torch_version.TorchVersion])` context manager to allowlist this global if you trust this class/function.

# Check the documentation of torch.load to learn more about types accepted by default with weights_only https://pytorch.org/docs/stable/generated/torch.load.html.


config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL torch.torch_version.TorchVersion was not an allowed global by default. Please use `torch.serialization.add_safe_globals([torch.torch_version.TorchVersion])` or the `torch.serialization.safe_globals([torch.torch_version.TorchVersion])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load 

### Get Diarization Results
Next, we process the file to get the timestamps where speech starts and ends.

In [8]:
# Process the audio file
audio_file = "./ami_samples/sample_0.wav"
print(f"Processing {audio_file} on {device}")
output = pipeline(audio_file)


Processing ./ami_samples/sample_0.wav on cpu


NameError: name 'pipeline' is not defined

The Pyannote Speaker Diarization model gives us a list of segment timestamps labeled with a speaker. 

In [ ]:
print("Voice activity segments:")

for segment, _, speaker in output.itertracks(yield_label = True):
        result = f"{segment.start:.2f} --> {segment.end:.2f} (duration: {segment.duration:.2f}s) Speaker: {speaker}"
        print(result)


### Additional Analytics

Now that we have processed our file we'll explore a few useful features of the Pyannote SDK:

1. See total speaker time broken down by speaker.
2. Find segments with speaker overlap (multiple speakers speaking at once).
3. Filter the data by speaker.

#### Speaker Time

Here we see the total speaking time for each speaker.

In [ ]:
for speaker in output.labels():
    speaking_time = output.label_duration(speaker)
    print(f"Speaker {speaker} total speaking time: {speaking_time:.2f}s")


#### Speaker Overlap

Pyannote shows us the timestamps where multiple speakers are speaking.

In [ ]:
overlap = output.get_overlap()
print(f"Overlapping speech regions: {overlap}")


#### Filter the Data by Speaker

We can use Pyannote to filter the output by speaker.

In [ ]:
speaker = "SPEAKER_06"
speaker_turns = output.label_timeline(speaker)
print(f"Speaker {speaker} speaks at:")
for speaker_turn in speaker_turns:
    print(speaker_turn)


## Inspect results

Next, we'll split the audio into chunks based on the diarization output in order to verify that it successfully isolated the speakers.


### Split the Audio

Here we write a function to split the original audio into segments determined by our Diarization output.

In [ ]:
import shutil
from pydub import AudioSegment

def split_audio_by_segments(audio_path, diarization_output, output_dir="output_segments"):
    """
    Split an audio file into multiple files based on diarization output
    
    Parameters:
    -----------
    audio_path: str
        Path to the input audio file
    diarization_output: Annotation
        Pyannote diarization output
    output_dir: str
        Directory to save the output segments
    """
    # Clear the output directory if it exists
    if os.path.exists(output_dir):
        shutil.rmtree(output_dir)
    
    # Create output directory
    os.makedirs(output_dir, exist_ok = True)
    
    # Load the audio file
    audio = AudioSegment.from_file(audio_path)
    
    # Extract each segment with speaker information
    for i, (segment, _, speaker) in enumerate(diarization_output.itertracks(yield_label = True)):
        # Convert seconds to milliseconds
        start_ms = int(segment.start * 1000)
        end_ms = int(segment.end * 1000)
        
        # Extract segment
        segment_audio = audio[start_ms:end_ms]
        
        # Generate output filename with speaker information
        filename = os.path.basename(audio_path)
        name, ext = os.path.splitext(filename)
        output_path = os.path.join(output_dir, f"{name}_segment_{i+1:04d}_{start_ms:08d}ms-{end_ms:08d}ms_speaker_{speaker}{ext}")
        
        # Export segment
        segment_audio.export(output_path, format = ext.replace('.', ''))
        print(f"Saved segment {i+1} to {output_path} (Speaker: {speaker})")


We then use that to save the segments to a local folder

In [ ]:
split_audio_by_segments(audio_file, output)


### Inspect Results

Here we create a function that allows us to play the audio files in our notebook.

In [ ]:
import librosa
from IPython.display import Audio, display

def play_audio(file_path, sr = None):
    """
    Play an audio file in a Jupyter notebook.
    
    Parameters:
    -----------
    file_path : str
        Path to the audio file to play
    sr : int, optional
        Sample rate to load the audio with. If None, uses the file's native sample rate.
        
    Returns:
    --------
    Audio widget that can be played in the notebook
    
    Example:
    --------
    >>> play_audio('path/to/audio.wav')
    """
    # Load the audio file
    y, sr = librosa.load(file_path, sr = sr)
    
    
    # Display an audio widget to play the sound
    audio_widget = Audio(data = y, rate = sr)
    display(audio_widget)


We'll use `play_audio` to listen to a few clips to verify that the speakers were correctly identified and isolated.

In [ ]:
import os
audio_dir = "./output_segments/"

audio_files = os.listdir(audio_dir)
audio_files.sort()

n_offset = 21
n_clips = 5

for fname in audio_files[n_offset:n_clips + n_offset]:
    print(f"File: {fname}")
    
    # Extract speaker info if present in filename
    if "_speaker_" in fname:
        speaker_part = fname.split("_speaker_")[1].split(".")[0]
        print(f"Speaker: {speaker_part}")
    
    play_audio(audio_dir + fname)


### Verify Speaker Overlap

Sometimes we find clips with multiple speakers speaking. We can check the overlap file to verify that there are multiple speakers speaking at that time. 

In [ ]:
overlap = output.get_overlap()
for overlap_ts in overlap:
    print(f"Overlapping speech regions: {overlap_ts}")


## Conclusion

The Pyannote speaker diarization model successfully identified multiple distinct speakers in the AMI Meeting Corpus sample. The model accurately detected overlapping speech regions, which we confirmed through our audio extraction and playback tests, demonstrating its effectiveness at handling complex conversational dynamics.
